# opencv_box — manual playground

Older exploration notebook for the **pre-contract** box (raw
`similarity_check` RPC, flat response status). The contract is now the
shared envelope:

* `Process` with `config = {"opencv": {"command": "match" | "similarity_check" | "reset", "parameters": {...}}}`
* namespaced response `{"opencv": {"status": "done|empty_request|error", ...}}`
* numpy fields declared `numpy` (np.save blobs — `np.load` to decode)

For scripted checks use `test_opencv.py` (against a running box) or
`smoke_inprocess.py` (no box build needed).


In [ ]:
import sys, json
sys.path.append("../protos")
import grpc
import pipeline_pb2, pipeline_pb2_grpc
from aux import wrap_value, unwrap_value
import numpy as np, io

def np_of(b): return np.load(io.BytesIO(bytes(b)), allow_pickle=False)

target = 'localhost:8061'
opts = [('grpc.max_send_message_length', -1), ('grpc.max_receive_message_length', -1)]
stub = pipeline_pb2_grpc.PipelineServiceStub(grpc.insecure_channel(target, options=opts))


In [ ]:
with open("00.jpg", "rb") as f: A = f.read()
with open("01.jpg", "rb") as f: B = f.read()

resp = stub.Process(pipeline_pb2.Envelope(
    config_json=json.dumps({"opencv": {"command": "match",
                                       "parameters": {"feature_extractor": "SIFT", "max_keypoints": 2000}}}),
    data={"images": wrap_value([A, B])}))
print(json.loads(resp.config_json))
for f in ("keypoints", "descriptors", "matches_inliers_a", "matches_inliers_b", "fundamental_matrix"):
    if f in resp.data:
        arr = np_of(unwrap_value(resp.data[f]))
        print(f, arr.shape, arr.dtype)


In [ ]:
# stateful frame gating — now a command, not a separate RPC
for frame in (A, B, A, B):
    r = stub.Process(pipeline_pb2.Envelope(
        config_json=json.dumps({"opencv": {"command": "similarity_check",
                                           "parameters": {"motion_thresh": 3.0}}}),
        data={"images": wrap_value([frame])}))
    print(json.loads(r.config_json))


In [ ]:
# clear the stateful gate again
r = stub.Process(pipeline_pb2.Envelope(config_json=json.dumps({"opencv": {"command": "reset"}})))
print(json.loads(r.config_json))
